# Script to get the differences of emissions between scenarios

This jupyter notebook contains all routine calculations of the emission differences in Europe compared to another scenario. 

**Authors:** Johannes Giehl (jfg.eco@cbs.dk)

## Import packages

In [1]:
#import
import pandas as pd
from cost_and_gas_source_share_functions import *

In [33]:
def filter_negative_supply(df, commodity_name):
    """
    Filters the DataFrame to a given commodity and removes rows with Supply >= 0.
    Handles column name capitalization issues.
    """
    # Standardize column names (optional safety step)
    df = df.rename(columns={col: col.strip().capitalize() for col in df.columns})
    
    # Check for required columns
    required_columns = {'Commodity', 'Node', 'Supply'}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"Missing required columns. Found columns: {df.columns.tolist()}")

    # Filter and return
    df_filtered = df[(df['Commodity'] == commodity_name) & (df['Supply'] < 0)].copy()

    # make values now positive for later emissions calculations
    df_filtered['Supply'] = df_filtered['Supply'] * -1  
    
    return df_filtered.reset_index(drop=True)


## Define path to data

In [14]:
#path for output and input data
#for gas shares per country
data_file_path = os.path.join('..', '..', '01_data', '02_output_data', '02_unidirectional_results', '01_paper_IAEE', '02_prepared_results')
full_data_file_path = os.path.abspath(os.path.join(os.getcwd(), data_file_path))
#for demand per country
demand_data_path = os.path.join('..', '..', '01_data', '01_input_data', '02_processed', '01_paper_IAEE')
full_demand_data_path = os.path.abspath(os.path.join(os.getcwd(), demand_data_path))

#specify the output file
output_file_name = '\emission_differences.xlsx'
output_file_path = data_file_path + output_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path))

## Load te data

In [15]:
#get all cost dfs from the excels in the prepared results folder
dfs = load_excel_sheets_by_name(full_data_file_path, "costs_shares", sheet_name="shares")
dfs_demand = load_excel_sheets_by_name(full_demand_data_path, "inputs_IAEE_2025_run", sheet_name="Supply")

## Prepare individual dataframes

In [12]:
df_names = list(dfs.keys())
df_names

['costs_shares_IAEE_2025_run_2021',
 'costs_shares_IAEE_2025_run_2024',
 'costs_shares_IAEE_2025_run_2024_inv',
 'costs_shares_IAEE_2025_run_2024_plus_no_QA',
 'costs_shares_IAEE_2025_run_2024_plus_NO_reduced',
 'costs_shares_IAEE_2025_run_2024_plus_no_USA',
 'costs_shares_IAEE_2025_run_2024_with_RU',
 'costs_shares_IAEE_2025_run_2035_AP',
 'costs_shares_IAEE_2025_run_2035_SP']

In [13]:
df_shares_IAEE_2025_run_2021 = dfs['costs_shares_IAEE_2025_run_2021']
df_shares_IAEE_2025_run_2024 = dfs['costs_shares_IAEE_2025_run_2024']
df_shares_IAEE_2025_run_2024_inv = dfs['costs_shares_IAEE_2025_run_2024_inv']
df_shares_IAEE_2025_run_2024_plus_no_QA = dfs['costs_shares_IAEE_2025_run_2024_plus_no_QA']
df_shares_IAEE_2025_run_2024_plus_NO_reduced = dfs['costs_shares_IAEE_2025_run_2024_plus_NO_reduced']
df_shares_IAEE_2025_run_2024_plus_no_USA = dfs['costs_shares_IAEE_2025_run_2024_plus_no_USA']
df_shares_IAEE_2025_run_2024_with_RU = dfs['costs_shares_IAEE_2025_run_2024_with_RU']
df_shares_IAEE_2025_run_2035_AP = dfs['costs_shares_IAEE_2025_run_2035_AP']
df_shares_IAEE_2025_run_2035_SP = dfs['costs_shares_IAEE_2025_run_2035_SP']

In [16]:
df_names_demand = list(dfs_demand.keys())
df_names_demand

['inputs_IAEE_2025_run_2021',
 'inputs_IAEE_2025_run_2024',
 'inputs_IAEE_2025_run_2024_inv',
 'inputs_IAEE_2025_run_2024_plus_no_QA',
 'inputs_IAEE_2025_run_2024_plus_NO_reduced',
 'inputs_IAEE_2025_run_2024_plus_no_USA',
 'inputs_IAEE_2025_run_2024_with_RU',
 'inputs_IAEE_2025_run_2035_AP',
 'inputs_IAEE_2025_run_2035_SP']

In [19]:
df_demand_IAEE_2025_run_2021                 = dfs_demand['inputs_IAEE_2025_run_2021']
df_demand_IAEE_2025_run_2024                 = dfs_demand['inputs_IAEE_2025_run_2024']
df_demand_IAEE_2025_run_2024_inv             = dfs_demand['inputs_IAEE_2025_run_2024_inv']
df_demand_IAEE_2025_run_2024_plus_no_QA      = dfs_demand['inputs_IAEE_2025_run_2024_plus_no_QA']
df_demand_IAEE_2025_run_2024_plus_NO_reduced = dfs_demand['inputs_IAEE_2025_run_2024_plus_NO_reduced']
df_demand_IAEE_2025_run_2024_plus_no_USA     = dfs_demand['inputs_IAEE_2025_run_2024_plus_no_USA']
df_demand_IAEE_2025_run_2024_with_RU         = dfs_demand['inputs_IAEE_2025_run_2024_with_RU']
df_demand_IAEE_2025_run_2035_AP              = dfs_demand['inputs_IAEE_2025_run_2035_AP']
df_demand_IAEE_2025_run_2035_SP              = dfs_demand['inputs_IAEE_2025_run_2035_SP']

In [36]:
#filter the dataframes so that they contain only demand
df_demand_2021_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2021, "Methane")
df_demand_2024_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024, "Methane")
df_demand_2024_inv_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_inv, "Methane")
df_demand_2024_no_QA_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_plus_no_QA, "Methane")
df_demand_2024_NO_red_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_plus_NO_reduced, "Methane")
df_demand_2024_no_USA_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_plus_no_USA, "Methane")
df_demand_2024_with_RU_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_with_RU, "Methane")
df_demand_2035_AP_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2035_AP, "Methane")
df_demand_2035_SP_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2035_SP, "Methane")